In [0]:
%load_ext autoreload
%autoreload 2

## Instalando bibliotecas

In [0]:
%pip install category_encoders scikit-learn matplotlib pandas statsmodels shap feature_engine lightgbm optuna

In [0]:
%pip freeze

In [0]:
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz


from warnings import filterwarnings
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
# from sklearn.model_selection import TunedThresholdClassifierCV  # This import may not exist, but leaving as is for now
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import itertools
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score, f1_score, make_scorer, accuracy_score, precision_score, recall_score, auc
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from category_encoders import TargetEncoder
from feature_engine.encoding import WoEEncoder, DecisionTreeEncoder
from category_encoders.woe import WOEEncoder
from sklearn.feature_selection import SelectFromModel, SelectKBest, f_classif
from sklearn.metrics import make_scorer, fbeta_score
import sys
from gold.util import *
import joblib
from sklearn.ensemble import HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
import pickle

filterwarnings('ignore')





## Lendo os dados

In [0]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
#202410
data_exec_inicial = 202503

# converte YYYYMM -> date
data_dt = datetime.strptime(str(data_exec_inicial), "%Y%m")

# subtrai 12 meses
data_exec_final = int((data_dt - relativedelta(months=5)).strftime("%Y%m"))
data_exec_final

In [0]:
path01 = "hackathon2025.silver.base_score_bureau_movel"
path02 = "hackathon2025.silver.book_pagamento"
path03 = "hackathon2025.silver.book_atraso"
path04 = "hackathon2025.silver.book_recarga"
path05 = "hackathon2025.silver.base_telco"
path06 = "hackathon2025.silver.base_dados_cadastrais"

In [0]:
df_bureau = (
    spark.read
         .table(path01)
         .filter(
             (col("SAFRA") >= data_exec_final) &
             (col("SAFRA") <= data_exec_inicial)
         )
)
df_bureau.count()

df_temp_01 = (
    df_bureau.alias("b")
    .join(
        df_recarga.alias("a"),
        on=["NUM_CPF", "SAFRA"],
        how="left"
    )
    .drop("rn", "DATPROC")
)

df_temp_01.createOrReplaceTempView("df_temp_01")

df_temp_01.count()


## Transformando o dataframe spark para pandas

In [0]:
abt_00 = df_bureau.toPandas()

In [0]:
abt_00 = abt_00.copy()
abt_01 = abt_00.drop(columns=["rn", "DATPROC"])
abt_01.head()

In [0]:
# Calcular a média do FPD e o volume por AAAAMM
resultado = abt_01.groupby('SAFRA').agg({'FPD': 'mean', 'SAFRA': 'count'}).rename(columns={'SAFRA': 'Volume'}).reset_index()
resultado.columns = ['Safra (AAAA)', 'Taxa_de_Evento', 'Volume']

# Exiba a tabela
resultado

In [0]:

df_tx_evento = plot_tx_event_volume_safra(abt_01,
                                              target='FPD',
                                              safra='SAFRA',
                                              ymax_volume=260000, ymax_taxa_evento=35)




## Validação Cruzada tipo Holdout utilizando modo out-of-time

   * Vamos utilizar as safras de 202410 a 202501 para desenvolvimento
   * Vamos utilizar as safras de 202502 a 202503 para validação do Modelo



In [0]:

# Filtrando a base de treino
abt_treino = abt_01[(abt_01['SAFRA'] >= '202410') & (abt_01['SAFRA'] <= '202501')]

# Filtrando a base de teste
abt_teste = abt_01[(abt_01['SAFRA'] >= '202502') & (abt_01['SAFRA'] <= '202503')]

abt_treino.shape,abt_teste.shape

## Dataprep

In [0]:
metadados = generate_metadata(abt_treino, ids=['NUM_CPF', 'SAFRA', 'FLAG_INSTALACAO', 'PROD', 'flag_mig2'], targets=['FPD'], orderby='PC_NULOS')
metadados.head(20)

In [0]:
# Base de treino
abt_treino = abt_01[
    (abt_01['SAFRA'] >= '202410') &
    (abt_01['SAFRA'] <= '202501')
].copy()

# Base de teste
abt_teste = abt_01[
    (abt_01['SAFRA'] >= '202502') &
    (abt_01['SAFRA'] <= '202503')
].copy()

abt_treino.shape, abt_teste.shape


In [0]:
# Colunas que NÃO entram no modelo
cols_drop = [
    'NUM_CPF',
    'SAFRA',
    'FLAG_INSTALACAO',
    'PROD',
    'flag_mig2',
    'FPD'
]

# Treino
X_train = abt_treino.drop(columns=cols_drop)
y_train = abt_treino['FPD']

# Teste
X_test = abt_teste.drop(columns=cols_drop)
y_test = abt_teste['FPD']


In [0]:
metadados = generate_metadata(X_train, ids=['NUM_CPF', 'SAFRA', 'FLAG_INSTALACAO', 'PROD', 'flag_mig2'], targets=['FPD'], orderby='PC_NULOS')
metadados.head(20)

In [0]:


from sklearn.preprocessing import OneHotEncoder


cat_features_low_card = metadados[(metadados['TIPO_FEATURE'] == 'object') & 
                                (metadados['CARDINALIDADE'] < 20)]['FEATURE'].tolist()

cat_features_high_card = metadados[(metadados['TIPO_FEATURE'] == 'object') & 
                                 (metadados['CARDINALIDADE'] >= 20)]['FEATURE'].tolist()

num_features = metadados[(metadados['TIPO_FEATURE'] != 'object')]['FEATURE'].tolist()

# Definir pipelines separados para cada tipo de feature categórica
cat_pipe_low = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

cat_pipe_high = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target_enc', TargetEncoder())
])

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# Combinar todos os pipelines
preprocessor = ColumnTransformer([
    ('cat_low', cat_pipe_low, cat_features_low_card),
    ('cat_high', cat_pipe_high, cat_features_high_card),
    ('num', num_pipe, num_features)
])

preprocesssor = Pipeline(steps=[("preprocessor", preprocessor)])

# Aplicar o pré-processamento
X_train_processed = preprocesssor.fit_transform(X_train, y_train)
X_test_processed = preprocesssor.transform(X_test)

# Para obter os nomes das colunas após o OneHotEncoder
# (isso é mais complexo pois o OneHotEncoder cria múltiplas colunas)
onehot_columns = []
if len(cat_features_low_card) > 0:
    onehot = preprocesssor.named_steps['preprocessor'].named_transformers_['cat_low'].named_steps['onehot']
    for i, col in enumerate(cat_features_low_card):
        cats = onehot.categories_[i]
        onehot_columns.extend([f"{col}_{cat}" for cat in cats])

# Nomes finais das colunas
processed_columns = onehot_columns + cat_features_high_card + num_features

# Converter para DataFrame
X_train_processed = pd.DataFrame(X_train_processed, columns=processed_columns)
X_test_processed = pd.DataFrame(X_test_processed, columns=processed_columns)



In [0]:
import os
os.makedirs('.gold/MODEL_02_SCORE1_e_2/artifacts/', exist_ok=True)
with open('.gold/MODEL_02_SCORE1_e_2/artifacts/prd_preprocesssor_skl.pkl', 'wb') as f:
  pickle.dump(preprocesssor, f)


## Feature Selection

In [0]:

clf = RandomForestClassifier(random_state=0, max_depth=5, min_samples_leaf=2)
clf.fit(X_train_processed, y_train)

In [0]:


#Obtendo o feature importance
feature_importances = clf.feature_importances_
features = pd.DataFrame({
    'Feature': X_train_processed.columns,
    'Importance': feature_importances
})

#Ordenar vars por importância
features = features.sort_values(by='Importance', ascending=False)

#Estabelecendo um ponto de corte
cutoff_maximp = 0.5

cutoff = cutoff_maximp * feature_importances.max()

# Selecionando vars acima do ponto de corte
selected_features = X_train_processed.columns[feature_importances > cutoff].tolist()
print('Número de features selecionadas: ', len(selected_features))

#Ordenar vars por importância
features = features.sort_values(by='Importance', ascending=True)

# Filtrar o DataFrame para apenas as features acima do corte
selected_features_df = features[features['Importance'] > cutoff]

# Ajusta o tamanho da figura com base no número de features selecionadas
plt.figure(figsize=(10, len(selected_features_df)*0.4))

# Plota as features selecionadas
plt.barh(selected_features_df['Feature'], selected_features_df['Importance'], color=(0.25, 0.5, 1))
plt.xlabel("Feature Importance")
plt.title("Variáveis Selecionadas - Random Forest")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()



In [0]:
import os
os.makedirs('.gold/MODEL_02_SCORE1_e_2/artifacts/', exist_ok=True)
# Salva as features selecionadas na pasta artifacts
with open('.gold/MODEL_02_SCORE1_e_2/artifacts/prd_selected_features_skl.pkl', 'wb') as f:
  pickle.dump(selected_features, f)

## Testar algoritmos

In [0]:


algoritmos = [
    DecisionTreeClassifier(criterion='gini', random_state=0, max_depth=7, min_samples_leaf=3),
    RandomForestClassifier(random_state=0, max_depth=7, min_samples_leaf=3),
    LGBMClassifier(random_state=0, max_depth=7, min_child_samples=3, n_jobs=-1, verbosity=-1,)
]

for algoritmo in algoritmos:

    nome_algoritmo = str(algoritmo)[:str(algoritmo).find("(")]
    # Treino do modelo
    algoritmo.fit(X_train_processed[selected_features],y_train)

    # Avaliar modelo
    metricas = calculate_metrics_models_classifier(nome_algoritmo,algoritmo, X_train_processed[selected_features], y_train, X_test_processed[selected_features], y_test)
    display(metricas)



## Tunagem de hiperparâmetros

In [0]:
import optuna

OPTUNA_EARLY_STOPING = 10

class EarlyStoppingExceeded(optuna.exceptions.OptunaError):
    early_stop = OPTUNA_EARLY_STOPING
    early_stop_count = 0
    best_score = None

def early_stopping_opt(study, trial):
    if EarlyStoppingExceeded.best_score == None:
      EarlyStoppingExceeded.best_score = study.best_value

    if study.best_value < EarlyStoppingExceeded.best_score:
        EarlyStoppingExceeded.best_score = study.best_value
        EarlyStoppingExceeded.early_stop_count = 0
    else:
      if EarlyStoppingExceeded.early_stop_count > EarlyStoppingExceeded.early_stop:
            EarlyStoppingExceeded.early_stop_count = 0
            best_score = None
            raise EarlyStoppingExceeded()
      else:
            EarlyStoppingExceeded.early_stop_count=EarlyStoppingExceeded.early_stop_count+1
    #print(f'EarlyStop counter: {EarlyStoppingExceeded.early_stop_count}, Best score: {study.best_value} and {EarlyStoppingExceeded.best_score}')
    return

In [0]:
# Definir CV
from sklearn.model_selection import StratifiedKFold, cross_val_score


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fbeta_scorer = make_scorer(
        fbeta_score,
        beta=1.3,          # dá mais peso para recall
        pos_label=1      # classe FPD
)


# Objetiva do Optuna
def objective(trial):


    # 1. DEFINIÇÃO DO ESPAÇO DE BUSCA DE HIPERPARÂMETROS
    # Cada trial do Optuna irá sortear uma combinação diferente

    params = {

        # Hiperparâmetros básicos
        'n_estimators': trial.suggest_int('n_estimators', 5, 50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.5, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 10, 100),
        
        # Regularização
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        
        # Amostragem
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        
        # Balanceamento de classes
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 2, 10),
        
        # Tipo de boosting

        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 5, 30),
        'boosting_type': trial.suggest_categorical('boosting_type', ['gbdt']),
        'objective': 'binary',
        'verbosity': -1,
        'random_state': 42
    }


    model = LGBMClassifier(**params)


    # cross_val_score retorna uma lista com a métrica para cada fold
    scores = cross_val_score(model, X_train_processed[selected_features], y_train, cv=cv, scoring=fbeta_scorer, n_jobs=-1)
    
    # Média dos AUCs
    return np.mean(scores)

# Estudo
study = optuna.create_study(direction="maximize",study_name="modelo")
study.add_trials(study.trials)
try:
    study.optimize(objective, n_trials=100, timeout=600, callbacks=[early_stopping_opt])

except EarlyStoppingExceeded:
    print(f'EarlyStopping Exceeded: No new best scores on iters {OPTUNA_EARLY_STOPING}')

print("Number of finished trials: {}".format(len(study.trials)))

print("Best trial:")
trial = study.best_trial

print("  Value: {}".format(trial.value))

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

In [0]:

algoritmo = LGBMClassifier(**study.best_params,random_state = 0)

nome_algoritmo = str(algoritmo)[:str(algoritmo).find("(")]
    # Treino do modelo
algoritmo.fit(X_train_processed[selected_features],y_train)

    # Avaliar modelo
metricas = calculate_metrics_models_classifier(nome_algoritmo,algoritmo, X_train_processed[selected_features], y_train, X_test_processed[selected_features], y_test)
display(metricas)

In [0]:
import os
os.makedirs('.gold/MODEL_02_SCORE1_e_2/artifacts/', exist_ok=True)
with open('.gold/MODEL_02_SCORE1_e_2/artifacts/modelo_lgbm.pkl', 'wb') as file:
  pickle.dump(algoritmo, file)

In [0]:
avaliar_modelo(X_train_processed[selected_features], y_train, X_test_processed[selected_features], y_test, algoritmo,nm_modelo='LGBM')

## Monitoramento do Modelo

O Índice de Estabilidade Populacional (Population Stability Index - PSI) é uma métrica utilizada para determinar a estabilidade de um modelo de score ao longo do tempo, comparando a distribuição dos scores em diferentes momentos (por exemplo, entre a distribuição dos scores em uma amostra de desenvolvimento e uma amostra de validação).

Um valor PSI elevado indica que a distribuição dos scores mudou significativamente entre as duas amostras, o que pode ser um sinal de que o modelo não é estável ao longo do tempo.



Os valores do PSI podem ser interpretados da seguinte forma:

  * PSI < 0,1: A mudança é insignificante, o que significa que a distribuição dos scores é muito semelhante entre os dois conjuntos de dados. O modelo é considerado estável.

  * 0,1 ≤ PSI < 0,25: Alguma mudança na distribuição dos scores foi detectada, mas geralmente é aceitável. Monitoramento é recomendado.

  * PSI ≥ 0,25: A mudança é significativa, indicando que a distribuição dos scores é consideravelmente diferente entre os dois conjuntos de dados. Isso pode ser um sinal de que o modelo não é mais adequado e pode precisar ser recalibrado ou reajustado.



In [0]:
import numpy as np

def calculate_psi(expected_scores, actual_scores, num_bins=10):
    """
    Calculate the Population Stability Index (PSI) between expected and actual scores.

    Parameters:
    - expected_scores: Scores from the development (or training) set.
    - actual_scores: Scores from the validation (or testing) set.
    - num_bins: Number of bins to use for score distributions.

    Returns:
    - psi_value: Calculated PSI value.
    """

    # Create bins for scores
    bin_edges = np.linspace(min(expected_scores.min(), actual_scores.min()),
                            max(expected_scores.max(), actual_scores.max()),
                            num_bins + 1)

    # Get the expected and actual proportions for each bin
    expected_proportions, _ = np.histogram(expected_scores, bins=bin_edges)
    actual_proportions, _ = np.histogram(actual_scores, bins=bin_edges)

    # Convert counts to proportions
    expected_proportions = expected_proportions / len(expected_scores)
    actual_proportions = actual_proportions / len(actual_scores)

    # Calculate PSI for each bin
    psi_bins = (actual_proportions - expected_proportions) * np.where(actual_proportions != 0,
                                                                       np.log(actual_proportions / expected_proportions),
                                                                       0)

    # Total PSI
    psi_value = np.sum(psi_bins)

    return psi_value

In [0]:
# Score do treino
y_train_prob = algoritmo.predict_proba(
    X_train_processed[selected_features]
)[:, 1]

# Score do teste / OOT
y_test_prob = algoritmo.predict_proba(
    X_test_processed[selected_features]
)[:, 1]


df_train_01 = pd.DataFrame({'Score_1': y_train_prob.round(4)})
abt_oot_01 = pd.DataFrame({'Score_1': y_test_prob.round(4)})

In [0]:

psi_value = calculate_psi(y_train_prob, y_test_prob, num_bins=10)
psi_value
